# Lab 6: Model Selection & Embedding-Backed Classification
**Course:** CIS 531/731 | **Module 6: GenAI & LLMs**

## Objective
Explore why specialized embedding models are often superior to generative LLMs for clustering and similarity-based classification. We will construct a 'gold standard' reference index of tropes, utilize cosine similarity and Euclidean distance for classification, and implement mathematical thresholds to handle 'Unknown' boundary zones.

### Environment Notes
Ensure `sentence-transformers` and `scipy` are installed in your DevContainer:
`!pip install sentence-transformers scipy numpy`

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from scipy.spatial.distance import cosine, euclidean
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully.")

### 1. Building the Reference Index
Instead of asking an LLM to guess a trope via zero-shot prompting, we construct a vector database (lookup table) of known tropes. We embed the core definition of each trope to serve as our mathematical anchors.

In [ ]:
# Define our gold standard reference tropes
reference_tropes = {
    "Chosen One Fantasy": "A protagonist is uniquely destined by prophecy or fate to save the world or defeat a great evil.",
    "Doomed Youth / Tragedy": "Young characters face inevitable death, trauma, or loss of innocence, often in war or disaster.",
    "Summer Romance": "A brief, intense romantic relationship that occurs during a summer vacation and inevitably ends.",
    "Sci-Fi Cyberpunk": "A dystopian future featuring advanced technology, artificial intelligence, cybernetics, and societal decay."
}

trope_names = list(reference_tropes.keys())
trope_descriptions = list(reference_tropes.values())

# Encode the reference index
reference_embeddings = model.encode(trope_descriptions)
print(f"Encoded {len(reference_embeddings)} reference tropes into the index.")

### 2. Distance Metrics for Classification
We map incoming, unknown text chunks into the same embedding space and calculate the distance to our reference anchors to assign a classification class.

In [ ]:
# Incoming ASR Chunks
corpus = [
    "The protagonist stared at the glowing sword, realizing it was the exact weapon described in the ancient prophecy.",
    "The caller sounded devastated, mentioning that the summer romance ended abruptly when they left for college.",
    "What passing-bells for these who die as cattle? Only the monstrous anger of the guns.",
    "I just want to order a pepperoni pizza with extra cheese for delivery."
]

print("Executing Embedding-Backed Classification...\n")

for i, text in enumerate(corpus):
    # Embed the unknown text
    query_embedding = model.encode([text])[0]
    
    # Calculate cosine similarities (1 - cosine distance)
    similarities = [1 - cosine(query_embedding, ref_emb) for ref_emb in reference_embeddings]
    
    # Find the best match
    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]
    best_trope = trope_names[best_match_idx]
    
    print(f"--- Transcript Chunk {i+1} ---")
    print(f"Text: '{text}'")
    print(f"Predicted Trope: {best_trope} (Cosine Similarity: {best_score:.4f})")
    print("----------------------------\n")

### 3. Thresholding and Confidence (Handling Out-of-Distribution Data)
Notice Chunk 4 (the pizza order) was classified as one of our literary tropes because the nearest-neighbor logic *had* to pick the closest vector. We prevent this by establishing a mathematical confidence threshold.

In [ ]:
CONFIDENCE_THRESHOLD = 0.35  # Threshold below which we flag as Unknown

print(f"Applying Confidence Threshold: {CONFIDENCE_THRESHOLD}\n")

for i, text in enumerate(corpus):
    query_embedding = model.encode([text])[0]
    similarities = [1 - cosine(query_embedding, ref_emb) for ref_emb in reference_embeddings]
    
    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]
    
    print(f"Text: '{text}'")
    # Apply boundary condition
    if best_score >= CONFIDENCE_THRESHOLD:
        print(f"-> Classified: {trope_names[best_match_idx]} ({best_score:.4f})")
    else:
        print(f"-> Classified: UNKNOWN / OUT OF DISTRIBUTION ({best_score:.4f} < Threshold)")
    print() 